Bearing RUL Prediction: Single LSTM Model with SHAP & Integrated Gradients
Research: Cross-Domain Bearing RUL Prediction under Shaft Misalignment

In [1]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
os.environ["OMP_NUM_THREADS"] = "1" 
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
# import shap
# from captum.attr import IntegratedGradients


In [2]:
torch.backends.cudnn.enabled = False
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# 1. GLOBAL CONFIGURATION

In [3]:
# 1. PERBAIKI KOLOM YANG DIBUANG
class Config:
    TRAIN_DATA_PATH = r"D:\ProyekDosen\Riset Bearing ShaftMissalignment + XAI\bearing_1\processed_bearing1.parquet"
    TEST_DATA_PATH  = r"D:\ProyekDosen\Riset Bearing ShaftMissalignment + XAI\bearing_2\processed_bearing2_synthetic_100db.parquet"
    OUTPUT_DIR      = r"D:\ProyekDosen\Riset Bearing ShaftMissalignment + XAI\results"
        
    COLS_TO_DROP = ['segment', 'time_s', 'time_min', 'bhi', 'label_vcd', 'T_cp', 'T_f',
                    'rms_ema_x', 'rms_ema_y', 'rms_ema_z']  # tambahkan ema cols juga
    TARGET_COL   = 'bhi'   # pastikan match exact dengan nama kolom parquet

    WINDOW_SIZE     = 300
    BATCH_SIZE      = 32
    HIDDEN_SIZE     = 64
    NUM_LAYERS      = 2
    LEARNING_RATE   = 0.001
    EPOCHS          = 50
    # DEVICE          = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    DEVICE          = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    BEARING_LIFE_S  = 392275

# Create output directory if it doesn't exist
os.makedirs(Config.OUTPUT_DIR, exist_ok=True)
print(f"Device: {Config.DEVICE}")

Device: cuda


In [4]:
"""Loads parquet, scales features, and creates sequence loaders"""
print("[INFO] Loading datasets...")
print(Config.TEST_DATA_PATH)
df_train = pd.read_parquet(Config.TRAIN_DATA_PATH)
df_test  = pd.read_parquet(Config.TEST_DATA_PATH)

[INFO] Loading datasets...
D:\ProyekDosen\Riset Bearing ShaftMissalignment + XAI\bearing_2\processed_bearing2_synthetic_100db.parquet


In [5]:
df_train.describe()

,segment,time_s,time_min,td_mean_x,td_mean_y,td_mean_z,td_std_x,td_std_y,td_std_z,td_rms_x,...,fd_bpfi_energy_x,fd_bpfi_energy_y,fd_bpfi_energy_z,fd_bsf_energy_x,fd_bsf_energy_y,fd_bsf_energy_z,rms_ema_x,rms_ema_y,rms_ema_z,bhi
count,6480.000000,6480.00000,6480.000000,6.480000e+03,6.480000e+03,6.480000e+03,6480.000000,6480.000000,6480.000000,6480.000000,...,6480.000000,6480.000000,6480.000000,6480.000000,6480.000000,6480.000000,6480.000000,6480.000000,6480.000000,6480.000000
mean,3239.500000,194370.00000,3239.500000,-3.296599e-07,-2.373796e-07,-7.167649e-07,9.177501,9.455964,4.864646,9.177470,...,0.472657,0.059060,0.004957,1.397487,0.067781,0.100751,9.177445,9.454498,4.864050,0.683719
std,1870.759204,112245.55225,1870.759204,4.396673e-04,4.002047e-04,1.441848e-03,0.432777,0.376575,0.156313,0.432776,...,0.133288,0.011973,0.002702,0.490092,0.037153,0.056936,0.227689,0.320974,0.125264,0.332961
min,0.000000,0.00000,0.000000,-1.076406e-02,-1.119694e-02,-1.996203e-02,8.264740,8.209218,4.141592,8.264713,...,0.282433,0.040766,0.001497,0.689243,0.016826,0.033605,8.545612,8.280481,4.514255,0.000000
25%,1619.750000,97185.00000,1619.750000,-2.442287e-04,-1.948595e-04,-7.013788e-04,8.839464,9.196243,4.752381,8.839433,...,0.384387,0.052291,0.002920,1.018433,0.037154,0.050049,9.036203,9.226411,4.771707,0.395254
50%,3239.500000,194370.00000,3239.500000,-1.547355e-06,2.264056e-06,-5.048910e-06,9.113097,9.409965,4.879557,9.113065,...,0.437371,0.057613,0.003599,1.220208,0.051524,0.084181,9.159316,9.441776,4.875845,0.790508
75%,4859.250000,291555.00000,4859.250000,2.410265e-04,1.907823e-04,6.756637e-04,9.463120,9.696164,4.970161,9.463092,...,0.539839,0.063509,0.007280,1.789454,0.105651,0.162105,9.315963,9.656830,4.948410,1.000000
max,6479.000000,388740.00000,6479.000000,7.990234e-03,8.302490e-03,2.343044e-02,11.742685,10.695141,7.225285,11.742639,...,1.813565,0.170247,0.022428,5.135437,0.209522,0.229086,10.052822,10.444613,5.270181,1.000000


In [6]:
df_test.describe()

,segment,time_s,time_min,td_mean_x,td_mean_y,td_mean_z,td_std_x,td_std_y,td_std_z,td_rms_x,...,fd_bsf_energy_x,fd_bsf_energy_y,fd_bsf_energy_z,rms_ema_x,rms_ema_y,rms_ema_z,bhi,label_vcd,T_cp,T_f
count,6480.000000,6480.00000,6480.000000,6.480000e+03,6.480000e+03,6.480000e+03,6480.000000,6480.000000,6480.000000,6480.000000,...,6480.000000,6480.000000,6480.000000,6480.000000,6480.000000,6480.000000,6480.000000,6480.000000,6480.0,6480.0
mean,3239.500000,194370.00000,3239.500000,-3.270898e-07,-2.400094e-07,-7.155261e-07,9.177501,9.455964,4.864646,9.177471,...,1.397487,0.067781,0.100751,9.177446,9.454498,4.864061,0.680247,0.639506,2336.0,6479.0
std,1870.759204,112245.55225,1870.759204,4.396757e-04,4.002035e-04,1.441848e-03,0.432777,0.376575,0.156313,0.432775,...,0.490092,0.037153,0.056936,0.227689,0.320974,0.125265,0.333121,0.480181,0.0,0.0
min,0.000000,0.00000,0.000000,-1.076446e-02,-1.119721e-02,-1.996170e-02,8.264735,8.209215,4.141589,8.264716,...,0.689244,0.016826,0.033605,8.545614,8.280481,4.514263,0.000000,0.000000,2336.0,6479.0
25%,1619.750000,97185.00000,1619.750000,-2.444072e-04,-1.948419e-04,-7.015567e-04,8.839465,9.196249,4.752380,8.839434,...,1.018433,0.037154,0.050049,9.036205,9.226412,4.771719,0.390961,0.000000,2336.0,6479.0
50%,3239.500000,194370.00000,3239.500000,-1.313525e-06,2.277533e-06,-5.233753e-06,9.113096,9.409968,4.879556,9.113063,...,1.220208,0.051524,0.084181,9.159316,9.441778,4.875857,0.781921,1.000000,2336.0,6479.0
75%,4859.250000,291555.00000,4859.250000,2.411156e-04,1.906465e-04,6.759028e-04,9.463119,9.696163,4.970162,9.463091,...,1.789454,0.105651,0.162105,9.315965,9.656831,4.948421,1.000000,1.000000,2336.0,6479.0
max,6479.000000,388740.00000,6479.000000,7.990315e-03,8.302837e-03,2.343035e-02,11.742685,10.695145,7.225284,11.742650,...,5.135438,0.209522,0.229086,10.052822,10.444611,5.270192,1.000000,1.000000,2336.0,6479.0


In [7]:
torch.backends.cudnn.enabled = False
torch.backends.cudnn.deterministic = True

# 2. DATA STRUCTURES & LOADERS

In [8]:
class BearingDataset(Dataset):
    def __init__(self, X, y, window_size, stride=1):
        self.X = X
        self.y = y
        self.window_size = window_size
        self.stride = stride
        self.indices = list(range(0, len(X) - window_size, stride))

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        start = self.indices[idx]
        end = start + self.window_size
        return torch.tensor(self.X[start:end], dtype=torch.float32), \
               torch.tensor([self.y[end-1]], dtype=torch.float32)

def load_and_preprocess_data():
    """Loads parquet, scales features, and creates sequence loaders"""
    print("[INFO] Loading datasets...")
    print(Config.TEST_DATA_PATH)
    df_train = pd.read_parquet(Config.TRAIN_DATA_PATH)
    df_test  = pd.read_parquet(Config.TEST_DATA_PATH)
    
    # Identify feature columns
    drop_cols = Config.COLS_TO_DROP + [Config.TARGET_COL]
    feature_cols = [c for c in df_train.columns if c not in drop_cols]
    print(f"[INFO] Detected {len(feature_cols)} input features.")
    
    # Extract Target
    y_train = df_train[Config.TARGET_COL].values
    y_test  = df_test[Config.TARGET_COL].values
    
    X_train = df_train[feature_cols].values 
    X_test  = df_test[feature_cols].values

    scaler  = MinMaxScaler()
    X_train = scaler.fit_transform(X_train)
    X_test  = scaler.transform(X_test)

    corr_with_bhi = [np.corrcoef(X_train[:, i], y_train)[0,1] for i in range(X_train.shape[1])]
    print(f"Max |corr| with BHI: {np.max(np.abs(corr_with_bhi)):.4f}")
    print(f"Mean |corr| with BHI: {np.mean(np.abs(corr_with_bhi)):.4f}")
    
    # Create DataLoaders
    train_dataset = BearingDataset(X_train, y_train, Config.WINDOW_SIZE, stride=1)
    test_dataset  = BearingDataset(X_test, y_test, Config.WINDOW_SIZE, stride=1)
    
    train_loader = DataLoader(train_dataset, batch_size=Config.BATCH_SIZE, shuffle=True)
    test_loader  = DataLoader(test_dataset, batch_size=Config.BATCH_SIZE, shuffle=False)
    
    return train_loader, test_loader, feature_cols, train_dataset, test_dataset

# 3. MODEL ARCHITECTURE


In [9]:
class RULLSTM(nn.Module):
    def __init__(self, input_size: int):
        super(RULLSTM, self).__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=128,  # Increased from 64
            num_layers=2,      # Increased from 2
            batch_first=True,
            dropout=0.0        # Increased dropout
        )
        self.fc1 = nn.Linear(128, 64)
        self.fc2 = nn.Linear(64, 1)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        last_timestep = lstm_out[:, -1, :]
        out = self.relu(self.fc1(last_timestep))
        out = self.sigmoid(self.fc2(out))
        return out


# 4. TRAINING & EVALUATION FUNCTIONS


In [10]:
def calculate_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    """Computes RUL/BHI evaluation metrics"""
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    
    # Relative Prediction Error (RPE) & Scoring Function
    epsilon = 1e-8
    rpe = np.mean(np.abs(y_true - y_pred) / (y_true + epsilon)) * 100
    
    # PRONOSTIA Asymmetric Scoring Function
    errors = y_pred - y_true
    score = np.sum(np.where(errors < 0, np.exp(-errors/13) - 1, np.exp(errors/10) - 1))
    
    return {'RMSE': rmse, 'MAE': mae, 'R2': r2, 'RPE_pct': rpe, 'Score': score}

def train_model(model, train_loader):
    criterion = nn.MSELoss(reduction='none')  # Changed to no reduction
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)  # Higher LR
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=7, min_lr=1e-5)
    
    loss_history = []
    print("\n[INFO] Starting Training...")
    
    for epoch in range(Config.EPOCHS):
        model.train()
        epoch_losses = []
        
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(Config.DEVICE), y_batch.to(Config.DEVICE)
            
            optimizer.zero_grad()
            outputs = model(X_batch)
            
            # Weighted loss - give more weight to degradation phase
            weights = torch.where(
                y_batch <  0.5,
                torch.tensor(3.0).to(Config.DEVICE),
                torch.tensor(1.0).to(Config.DEVICE)
            )
            loss = (criterion(outputs, y_batch) * weights).mean()
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            epoch_losses.append(loss.item())
            
        avg_loss = np.mean(epoch_losses)
        loss_history.append(avg_loss)
        scheduler.step(avg_loss)
        
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Epoch [{epoch+1}/{Config.EPOCHS}] | Loss: {avg_loss:.6f} | LR: {optimizer.param_groups[0]['lr']:.6f}")
            
    return loss_history

def evaluate_and_export(model, test_loader):
    print(Config.TEST_DATA_PATH)
    """Evaluates the model and exports predictions to Excel"""
    model.eval()
    predictions, targets = [], []
    
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(Config.DEVICE)
            preds = model(X_batch).cpu().numpy()
            predictions.extend(preds.flatten())
            targets.extend(y_batch.numpy().flatten())
            
    predictions = np.array(predictions)
    targets = np.array(targets)
    
    metrics = calculate_metrics(targets, predictions)
    print("\n[INFO] Evaluation Metrics on Bearing-2 (Test Set):")
    for k, v in metrics.items():
        print(f" - {k}: {v:.4f}")
        
    # Convert BHI to RUL (Simple linear conversion for evaluation)
    true_rul = targets * Config.BEARING_LIFE_S
    pred_rul = predictions * Config.BEARING_LIFE_S
    
    # Export to Excel
    df_export = pd.DataFrame({
        'Time_Step': np.arange(len(targets)),
        'True_BHI': targets,
        'Predicted_BHI': predictions,
        'True_RUL_Seconds': true_rul,
        'Predicted_RUL_Seconds': pred_rul
    })
    
    export_path = os.path.join(Config.OUTPUT_DIR, "Predictions_Results.xlsx")
    df_export.to_excel(export_path, index=False)
    print(f"[INFO] Predictions saved successfully to: {export_path}")
    
    return targets, predictions

# 5. VISUALIZATIONS & XAI (SHAP & CAPTUM)


In [11]:
def plot_rul(y_true: np.ndarray, y_pred: np.ndarray):
    """Plots actual vs predicted Health Index"""
    plt.figure(figsize=(12, 5))
    plt.plot(y_true, label='True Health Index (BHI)', color='blue', linewidth=2)
    plt.plot(y_pred, label='Predicted Health Index', color='red', linestyle='--', linewidth=2)
    plt.title('Bearing Health Index (BHI) Prediction on Shaft Misalignment Data')
    plt.xlabel('Time Steps')
    plt.ylabel('Health Index (1 = Healthy, 0 = Failed)')
    plt.legend()
    plt.grid(alpha=0.4)
    plt.tight_layout()
    plt.savefig(os.path.join(Config.OUTPUT_DIR, "BHI_Prediction_Plot.png"), dpi=300)
    plt.show()

# def run_xai(model, test_dataset, feature_names):
#     """Applies SHAP and Integrated Gradients for Model Interpretability"""
#     print("\n[INFO] Initializing Explainable AI (XAI) Analysis...")
#     model.eval()
    
#     # Sample background and test instances for XAI (Memory optimization)
#     background_tensor = test_dataset.X[:100].to(Config.DEVICE)
#     test_tensor = test_dataset.X[100:105].to(Config.DEVICE)
    
#     # --- 1. CAPTUM (Integrated Gradients) ---
#     print("[INFO] Executing Integrated Gradients (Captum)...")
#     ig = IntegratedGradients(model)
#     attributions, delta = ig.attribute(test_tensor, target=None, return_convergence_delta=True)
    
#     # Summarize attributions across the time window (dim 1)
#     attr_sum = attributions.cpu().detach().numpy().sum(axis=1).mean(axis=0)
    
#     # --- 2. SHAP (GradientExplainer) ---
#     print("[INFO] Executing SHAP GradientExplainer...")
#     explainer = shap.GradientExplainer(model, background_tensor)
#     shap_values = explainer.shap_values(test_tensor)
    
#     # SHAP returns a list of arrays for PyTorch, we need to average across timesteps
#     shap_val_2d = shap_values[0].mean(axis=1) if isinstance(shap_values, list) else shap_values.mean(axis=1)
    
#     # --- Visualizing XAI ---
#     plt.figure(figsize=(10, 8))
#     # Plot top 15 features based on Integrated Gradients
#     feat_importances = pd.Series(np.abs(attr_sum), index=feature_names)
#     feat_importances.nlargest(15).sort_values().plot(kind='barh', color='teal')
#     plt.title("Top 15 Dominant Features (Integrated Gradients)")
#     plt.xlabel("Absolute Attribution Magnitude")
#     plt.tight_layout()
#     plt.savefig(os.path.join(Config.OUTPUT_DIR, "XAI_IntegratedGradients_Plot.png"), dpi=300)
#     plt.show()

# MAIN EXECUTION


In [ ]:
if __name__ == "__main__":
    # 1. Load Data
    train_loader, test_loader, features, _, test_ds = load_and_preprocess_data()
    
    # 2. Initialize Model
    model = RULLSTM(input_size=len(features)).to(Config.DEVICE)
    
    # 3. Train
    loss_hist = train_model(model, train_loader)
    
    # 4. Evaluate & Export
    y_true, y_pred = evaluate_and_export(model, test_loader)
    plot_rul(y_true, y_pred)
    
    # 5. Interpret (XAI)
    # run_xai(model, test_ds, features)
    
    print("\n[SUCCESS] Pipeline Executed Successfully!")

[INFO] Loading datasets...
D:\ProyekDosen\Riset Bearing ShaftMissalignment + XAI\bearing_2\processed_bearing2_synthetic_100db.parquet
[INFO] Detected 54 input features.
Max |corr| with BHI: 0.5282
Mean |corr| with BHI: 0.1906


d:\torch_env\Lib\site-packages\torch\nn\modules\module.py:1341: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:28.)
  return t.to(



[INFO] Starting Training...
Epoch [1/50] | Loss: 0.081277 | LR: 0.001000
Epoch [5/50] | Loss: 0.016007 | LR: 0.001000
Epoch [10/50] | Loss: 0.001798 | LR: 0.001000
Epoch [15/50] | Loss: 0.000785 | LR: 0.001000


In [ ]:
# 4. Evaluate & Export
def load_and_preprocess_data():
    """Loads parquet, scales features, and creates sequence loaders"""
    print("[INFO] Loading datasets...")
    print(Config.TEST_DATA_PATH)
    df_train = pd.read_parquet(Config.TRAIN_DATA_PATH)
    df_test  = pd.read_parquet(Config.TEST_DATA_PATH)
    
    # Identify feature columns
    drop_cols = Config.COLS_TO_DROP + [Config.TARGET_COL]
    feature_cols = [c for c in df_train.columns if c not in drop_cols]
    print(f"[INFO] Detected {len(feature_cols)} input features.")
    
    # Extract Target
    y_train = df_train[Config.TARGET_COL].values
    y_test  = df_test[Config.TARGET_COL].values
    
    X_train = df_train[feature_cols].values 
    X_test  = df_test[feature_cols].values

    corr_with_bhi = [np.corrcoef(X_train[:, i], y_train)[0,1] for i in range(X_train.shape[1])]
    print(f"Max |corr| with BHI: {np.max(np.abs(corr_with_bhi)):.4f}")
    print(f"Mean |corr| with BHI: {np.mean(np.abs(corr_with_bhi)):.4f}")
    
    # Create DataLoaders
    train_dataset = BearingDataset(X_train, y_train, Config.WINDOW_SIZE, stride=1)
    test_dataset  = BearingDataset(X_test, y_test, Config.WINDOW_SIZE, stride=1)
    
    train_loader = DataLoader(train_dataset, batch_size=Config.BATCH_SIZE, shuffle=True)
    test_loader  = DataLoader(test_dataset, batch_size=Config.BATCH_SIZE, shuffle=False)
    
    return train_loader, test_loader, feature_cols, train_dataset, test_dataset

TEST_DATA_PATH  = r"D:\ProyekDosen\Riset Bearing ShaftMissalignment + XAI\bearing_1\processed_bearing1.parquet"
train_loader, test_loader, features, _, test_ds = load_and_preprocess_data()
y_true, y_pred = evaluate_and_export(model, test_loader)
plot_rul(y_true, y_pred)